# Deep Joint Multi-Modal Yelp Fake Detector — CONTRASTIVE (SupCon, tuned, 642k, no AI)

Ablation variant of Model B's BiLSTM member: same GloVe+BiLSTM text branch ⊕ behavioural MLP
fusion, but trained with a **supervised contrastive** auxiliary objective (Khosla 2020) jointly
with cross-entropy (Gunel 2021). **The standard Model B is untouched** — a separate detector for
the focal-vs-contrastive-vs-standard comparison.

* **Text branch** — GloVe(300d) + BiLSTM(128) → 64-d
* **Behavioural branch** — MLP over the 17 engineered features → 32-d
* **Fusion** — concat → 64-d fused rep → logit → sigmoid (the served path, exported to ONNX)
* **Contrastive head (train only)** — fused → proj → 128-d on the unit sphere; SupCon pulls
  same-label reps together / pushes different-label apart. **Not exported** — serving is a
  drop-in for standard Model B.
* **Training** — custom GradientTape loop, balanced 50/50 batches so SupCon always sees both
  classes (train is the true ~11% fake). Loss `(1-λ)·CE + λ·SupCon`. Early stopping on val
  macro-F1 with best-weight restore.

**Best-shot / tuned:** λ (dominant CE↔SupCon mix) **and** τ (temperature) are **swept** —
**Phase A** tunes (λ, τ) on a fast proxy (a 150k stratified subsample, balanced batches, capped
epochs, scored on the full balanced val set), then **Phase B** retrains from scratch at the best
config on the **full** train. The chosen (λ, τ) are recorded in the meta.

**Expectation (honest):** the text-only contrastive ablation landed *below* baseline, and SupCon
only reshapes the embedding geometry (adds no new signal beyond the fused features), so even tuned
this may not beat standard Model B. The sweep gives it a genuine best shot for the ablation table.

**Before running:** upload `data/yelp_multimodal_features_full.csv` (~480 MB) to your Drive root;
set runtime to **GPU** (high-RAM recommended — 642k rows). **Output:** downloads
`yelp_multimodal_lstm_contrastive_onnx.zip` (ONNX + tokenizer + scaler + threshold) → extract to
`data/yelp_multimodal_lstm_contrastive_onnx/`. Feature order matches `model_b.FEAT_COLS`.
**Reference (same Yelp test):** text-only ~69% · standard Model B (full) ens 76.2% / xgb 77.3% / DL ~76.7%.

In [ ]:
# GloVe 300d (skip if already present)
import os
if not os.path.exists('glove.6B.300d.txt'):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.300d.txt
print('GloVe ready:', os.path.exists('glove.6B.300d.txt'))

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

CSV     = '/content/drive/MyDrive/yelp_multimodal_features_full.csv'   # PURE full 678k (no AI)
MAX_LEN = 250
VOCAB   = 30000
EMBED   = 300
DROP    = {'user_reviews_on_this_biz'}
META    = {'text','label','split','source','fake_type','generator','length_bucket'}

df = pd.read_csv(CSV)
df['text'] = df['text'].astype(str)
feat_cols = [c for c in df.columns if c not in META and c not in DROP]
print('rows', len(df), '| behavioural features', len(feat_cols))
# fake rate per split: train is the true ~11% (imbalanced), val/test balanced 50/50
print(df.groupby('split')['label'].agg(n='size', fake_rate='mean'))

In [ ]:
# splits
tr = df[df.split=='train'].reset_index(drop=True)
va = df[df.split=='val'].reset_index(drop=True)
te = df[df.split=='test'].reset_index(drop=True)
y_tr, y_va, y_te = tr.label.values, va.label.values, te.label.values

# --- TEXT modality: tokenize + pad ---
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tok = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tok.fit_on_texts(tr.text.values)
def seq(s): return pad_sequences(tok.texts_to_sequences(s), maxlen=MAX_LEN, padding='post', truncating='post')
Xtr_t, Xva_t, Xte_t = seq(tr.text.values), seq(va.text.values), seq(te.text.values)

# --- BEHAVIOURAL modality: standardize (fit on train) ---
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
Xtr_b = sc.fit_transform(tr[feat_cols].values).astype('float32')
Xva_b = sc.transform(va[feat_cols].values).astype('float32')
Xte_b = sc.transform(te[feat_cols].values).astype('float32')
print('text', Xtr_t.shape, '| behavioural', Xtr_b.shape)

In [ ]:
# GloVe embedding matrix
emb_index = {}
with open('glove.6B.300d.txt', encoding='utf-8') as f:
    for line in f:
        v = line.split()
        emb_index[v[0]] = np.asarray(v[1:], dtype='float32')
emb_matrix = np.zeros((VOCAB, EMBED), dtype='float32')
hit = 0
for w, i in tok.word_index.items():
    if i < VOCAB:
        e = emb_index.get(w)
        if e is not None:
            emb_matrix[i] = e; hit += 1
print('GloVe coverage:', hit, '/', min(len(tok.word_index), VOCAB))

In [ ]:
# Model builder (CONTRASTIVE variant) + SupCon loss. Same fusion architecture as standard
# Model B, plus a SupCon projection head on the FUSED representation (training only). Two views
# of one shared graph, rebuilt fresh per config in the sweep:
#   * model      = [text,beh] -> sigmoid prob   (served path, exported to ONNX; identical I/O to Model B)
#   * proj_model = [text,beh] -> (logit, L2-normed 128-d embedding)  (used by the custom loop)
from tensorflow.keras.layers import (Input, Embedding, Bidirectional, LSTM,
    SpatialDropout1D, Dense, Dropout, Concatenate, Activation, Lambda)
from tensorflow.keras.models import Model

def build_models():
    # NOTE: do NOT call keras.backend.clear_session() here. On Colab GPU (Keras 3) it invalidates
    # the CUDA handle used by the seed generator, and the next Embedding weight-init throws
    # CUDA_ERROR_INVALID_HANDLE. Each build already produces an independent model with unique
    # layer names; at most ~2 are alive at once in the sweep, so there is no leak.
    t_in = Input(shape=(MAX_LEN,), name='text_input')
    x = Embedding(VOCAB, EMBED, weights=[emb_matrix], trainable=False)(t_in)   # input_length deprecated -> removed
    x = SpatialDropout1D(0.3)(x)
    x = Bidirectional(LSTM(128))(x)
    x = Dropout(0.4)(x)
    text_repr = Dense(64, activation='relu')(x)
    b_in = Input(shape=(len(feat_cols),), name='beh_input')
    b = Dense(64, activation='relu')(b_in)
    b = Dropout(0.3)(b)
    beh_repr = Dense(32, activation='relu')(b)
    f = Concatenate()([text_repr, beh_repr])
    fused = Dense(64, activation='relu', name='fused')(f)
    fused = Dropout(0.4)(fused)
    logit = Dense(1, name='logit')(fused)                 # raw logit (BCE-from-logits in the loop)
    out   = Activation('sigmoid', name='prob')(logit)     # served probability (drop-in for Model B)
    pj = Dense(128, activation='relu')(fused)             # SupCon projection (train only; not exported)
    pj = Dense(128)(pj)
    proj_emb = Lambda(lambda z: tf.math.l2_normalize(z, axis=1), name='proj_emb')(pj)
    model      = Model([t_in, b_in], out)
    proj_model = Model([t_in, b_in], [logit, proj_emb])
    return model, proj_model

# Supervised contrastive loss (Khosla 2020) on L2-normed fused embeddings; joint with CE
# (Gunel 2021): loss = (1-lambda)*CE + lambda*SupCon. lambda AND temperature are SWEPT (cell-6).
def supcon_loss(emb, labels, temperature):
    labels = tf.cast(tf.reshape(labels, [-1, 1]), tf.float32)
    B = tf.shape(emb)[0]
    sim = tf.matmul(emb, emb, transpose_b=True) / temperature
    sim = sim - tf.stop_gradient(tf.reduce_max(sim, axis=1, keepdims=True))   # stability
    logits_mask = 1.0 - tf.eye(B)                                             # exclude self
    pos_mask = tf.cast(tf.equal(labels, tf.transpose(labels)), tf.float32) * logits_mask
    exp_sim = tf.exp(sim) * logits_mask
    log_prob = sim - tf.math.log(tf.reduce_sum(exp_sim, axis=1, keepdims=True) + 1e-12)
    pos_cnt = tf.reduce_sum(pos_mask, axis=1)
    mean_log_prob = tf.reduce_sum(pos_mask * log_prob, axis=1) / tf.maximum(pos_cnt, 1.0)
    valid = tf.cast(pos_cnt > 0, tf.float32)                                  # anchors w/ >=1 positive
    return -tf.reduce_sum(mean_log_prob * valid) / tf.maximum(tf.reduce_sum(valid), 1.0)
print('build_models + supcon_loss ready')

In [ ]:
# Custom training loop (SupCon needs a GradientTape + balanced batches) wrapped in a reusable
# fit_once so the sweep and the final train share one code path. Each step is a 50/50 batch (train
# is the true ~11% fake); CE is plain mean (the balance replaces class weights). Early stop on val
# macro-F1, restore best weights. fit_once builds a FRESH model+optimizer+train_step per config.
from sklearn.metrics import f1_score
import itertools

BATCH = 256; HALF = BATCH // 2

def fit_once(lam, temp, Xt, Xb, y, epochs, patience, tag='', verbose=False):
    model, proj_model = build_models()
    opt = tf.keras.optimizers.Adam(1e-3)
    pos_idx = np.where(y == 1)[0]; neg_idx = np.where(y == 0)[0]
    n_batches = len(neg_idx) // HALF
    @tf.function
    def train_step(xt, xb, yy):
        with tf.GradientTape() as tape:
            logit, emb = proj_model([xt, xb], training=True)
            logit = tf.reshape(logit, [-1])
            ce = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=yy, logits=logit))
            scl = supcon_loss(emb, yy, temp)
            loss = (1.0 - lam) * ce + lam * scl
        grads = tape.gradient(loss, proj_model.trainable_variables)
        opt.apply_gradients(zip(grads, proj_model.trainable_variables))
        return loss, ce, scl
    def val_macro_f1():
        p = model.predict([Xva_t, Xva_b], verbose=0).reshape(-1)
        bf, bt = -1, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            s = f1_score(y_va, (p >= t).astype(int), average='macro')
            if s > bf: bf, bt = s, t
        return bf, bt
    lh, vh = [], []
    best_f1, best_w, best_t, best_ep, wait = -1, None, 0.5, 0, 0
    for ep in range(epochs):
        neg = np.random.permutation(neg_idx)
        pos = np.random.choice(pos_idx, size=n_batches * HALF, replace=True)
        tot = tot_ce = tot_sc = 0.0
        for bi in range(n_batches):
            sl = slice(bi * HALF, (bi + 1) * HALF)
            idx = np.concatenate([neg[sl], pos[sl]]); np.random.shuffle(idx)
            xt = tf.convert_to_tensor(Xt[idx]); xb = tf.convert_to_tensor(Xb[idx])
            yy = tf.convert_to_tensor(y[idx].astype('float32'))
            l, c, s = train_step(xt, xb, yy); tot += float(l); tot_ce += float(c); tot_sc += float(s)
        f1, thr = val_macro_f1(); lh.append(tot / n_batches); vh.append(f1)
        if verbose:
            print(f'  {tag} ep{ep+1}: ce={tot_ce/n_batches:.4f} sup={tot_sc/n_batches:.4f}  val_macroF1={f1:.4f} @thr {thr:.2f}')
        if f1 > best_f1:
            best_f1, best_t, best_ep = f1, thr, ep + 1; best_w = model.get_weights(); wait = 0
        else:
            wait += 1
            if wait >= patience: break
    model.set_weights(best_w)
    return model, best_t, best_f1, best_ep, lh, vh

# ============================ PHASE A — hyperparameter sweep ============================
PROXY_N      = 150_000
PROXY_EPOCHS = 8
PROXY_PAT    = 2
LAMBDAS = [0.1, 0.3, 0.5]          # contrastive weight (dominant knob)
TEMPS   = [0.1, 0.3]               # SupCon temperature

rng = np.random.default_rng(42)
idx = rng.choice(len(y_tr), size=min(PROXY_N, len(y_tr)), replace=False)
pXt, pXb, py = Xtr_t[idx], Xtr_b[idx], y_tr[idx]
print(f'PROXY sweep: {len(idx)} rows (fake rate {py.mean():.3f}, balanced batches) | '
      f'{len(LAMBDAS)*len(TEMPS)} configs x up to {PROXY_EPOCHS} epochs\n')

sweep=[]
for lam,temp in itertools.product(LAMBDAS, TEMPS):
    _, _, f1, bep, _, _ = fit_once(lam, temp, pXt, pXb, py, PROXY_EPOCHS, PROXY_PAT,
                                   tag=f'[L{lam} t{temp}]', verbose=False)
    sweep.append((f1, lam, temp))
    print(f'  config lambda={lam} tau={temp}  ->  proxy val macroF1={f1:.4f} (ep {bep})')
sweep.sort(reverse=True)
print('\nPROXY ranking (best first):')
for f1,lam,temp in sweep: print(f'   lambda={lam} tau={temp}: {f1:.4f}')
BEST_F1, BEST_LAMBDA, BEST_TEMP = sweep[0]
print(f'\n==> BEST proxy config: lambda={BEST_LAMBDA}, tau={BEST_TEMP} (proxy val {BEST_F1:.4f})')

# ============================ PHASE B — final full-data train ============================
print('\n=============== FINAL: full-data train @ best config ===============')
model, best_thr, final_f1, best_ep, loss_hist, val_hist = fit_once(
    BEST_LAMBDA, BEST_TEMP, Xtr_t, Xtr_b, y_tr, 30, 5, tag='FINAL', verbose=True)
SUPCON_LAMBDA, SUPCON_TEMP = BEST_LAMBDA, BEST_TEMP   # recorded into meta on export
print(f'FINAL: best epoch {best_ep}  val macroF1={final_f1:.4f} @thr {best_thr:.2f} '
      f'(lambda={SUPCON_LAMBDA}, tau={SUPCON_TEMP})')

In [ ]:
# Training curves (custom loop history)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,len(loss_hist)+1), loss_hist, marker='o'); ax[0].set_title('Train loss (CE + λ·SupCon)'); ax[0].set_xlabel('epoch')
ax[1].plot(range(1,len(val_hist)+1), val_hist, marker='o', color='green', label='val macro-F1')
ax[1].axvline(best_ep, ls='--', color='red', label=f'best epoch {best_ep}')
ax[1].set_title('Val macro-F1'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig('yelp_multimodal_lstm_training.png', dpi=120); plt.show()

In [ ]:
# Test evaluation (val-tuned threshold) — pure human-fake task, no per-AI-type breakdown
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
thr  = best_thr
p_te = model.predict([Xte_t, Xte_b], verbose=0).reshape(-1)
pred = (p_te >= thr).astype(int)
print('threshold (val-tuned macroF1) =', round(thr, 2))
print('test accuracy  =', round(accuracy_score(y_te, pred), 4))
print('test macro-F1  =', round(f1_score(y_te, pred, average='macro'), 4))
print('fake recall    =', round(recall_score(y_te, pred, pos_label=1), 3))
print('fake precision =', round(precision_score(y_te, pred, pos_label=1), 3))
print('genuine spec   =', round(recall_score(y_te, pred, pos_label=0), 3))
print('\nReferences (same Yelp test): text-only ~69% · pure sklearn Model B (full) ens 76.2% / xgb 77.3% · standard Model B DL ~76.7%')
print()
print(classification_report(y_te, pred, target_names=['genuine','fake'], digits=4))

In [ ]:
# Save artifacts (keras for CPU re-export, + scaler/tokenizer/threshold for serving).
# `model` is the sigmoid serving head only (the SupCon proj head is dropped) — identical I/O to
# standard Model B, so the export below is a drop-in.
import json, joblib
THR=float(best_thr)
model.save('yelp_multimodal_lstm.keras')
joblib.dump(sc, 'yelp_multimodal_lstm_scaler.joblib')          # serving must standardise behaviour
with open('yelp_multimodal_lstm_tokenizer.json','w',encoding='utf-8') as f: f.write(tok.to_json())
json.dump({'threshold':THR,'feat_cols':feat_cols,'max_len':MAX_LEN,'vocab':VOCAB,
           'supcon_lambda':float(SUPCON_LAMBDA),'supcon_tau':float(SUPCON_TEMP)},
          open('yelp_multimodal_lstm_meta.json','w'), indent=2)
print('saved keras + scaler + tokenizer + meta | threshold', THR, '| supcon', SUPCON_LAMBDA, SUPCON_TEMP)

In [ ]:
# Export to ONNX (thread-safe serving, like the project's other DL models).
# Re-export the SavedModel with CUDA hidden so TF uses standard LSTM ops, not
# CudnnRNNV3 (which ONNX Runtime can't run). Multi-input: text_input + beh_input.
# compile=False: this model was trained with a custom GradientTape loop (no compiled loss).
!pip install -q tf2onnx onnxruntime
import subprocess, os
open('cpu_export.py','w').write(
'''import os
os.environ["CUDA_VISIBLE_DEVICES"]=""
import tensorflow as tf
m=tf.keras.models.load_model("yelp_multimodal_lstm.keras", compile=False)
m.export("yelp_multimodal_lstm_savedmodel")
print("CPU SavedModel export done")
''')
r1=subprocess.run(['python','cpu_export.py'],capture_output=True,text=True,
                  env={**os.environ,'CUDA_VISIBLE_DEVICES':''})
print(r1.stdout);  print('STDERR:',r1.stderr[-800:]) if r1.returncode else None
r2=subprocess.run(['python','-m','tf2onnx.convert','--saved-model','yelp_multimodal_lstm_savedmodel',
                   '--output','yelp_multimodal_lstm.onnx','--opset','13'],capture_output=True,text=True)
print(r2.stdout[-400:]); print('STDERR:',r2.stderr[-1200:]) if r2.returncode else None

In [ ]:
# Sanity check: ONNX outputs must match Keras (feed both inputs, mapped by last dim)
import onnxruntime as ort, numpy as np
sess=ort.InferenceSession('yelp_multimodal_lstm.onnx', providers=['CPUExecutionProvider'])
print('ONNX inputs:', [(i.name,i.shape) for i in sess.get_inputs()])
feed={}
for i in sess.get_inputs():
    feed[i.name]=(Xte_t[:4].astype(np.float32) if i.shape[-1]==MAX_LEN else Xte_b[:4].astype(np.float32))
onnx_p=sess.run(None, feed)[0].flatten()
keras_p=model.predict([Xte_t[:4],Xte_b[:4]],verbose=0).flatten()
print('ONNX :', np.round(onnx_p,4))
print('Keras:', np.round(keras_p,4))
print('match:', bool(np.allclose(onnx_p,keras_p,atol=1e-3)))

In [ ]:
# Bundle + download -> extract to data/yelp_multimodal_lstm_contrastive_onnx/
import zipfile
from google.colab import files
with zipfile.ZipFile('yelp_multimodal_lstm_contrastive_onnx.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['yelp_multimodal_lstm.onnx','yelp_multimodal_lstm_tokenizer.json',
               'yelp_multimodal_lstm_scaler.joblib','yelp_multimodal_lstm_meta.json']:
        zf.write(fn)
print('created yelp_multimodal_lstm_contrastive_onnx.zip  (extract -> data/yelp_multimodal_lstm_contrastive_onnx/)')
files.download('yelp_multimodal_lstm_contrastive_onnx.zip')